In [1]:
#### mv packages ####
import modules.data as d
import modules.model as m
import modules.pooling as p
import modules.train as t
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device, generator = u.Devices().auto_set_device()#['cuda:1', 'cuda:0'])
# device, generator = u.Devices().set_device('cpu')

#### data ####
brca = d.Preprocessor(
    tcga_project='TCGA-BRCA',
    tcga_dir=dataset_dir/'tcga',
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',
    
    # counts
    apply_DESeq_norm=False, 
    log_transform=False,
    scale_method=None,

    # etc
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Primary Tumor']},
    max_subset = 120,
)
_dataset = d.GraphDataset(brca)
_batch = d.get_toy_databatch(_dataset, generator)

# #### Device() ####
# device = cuda:4

# #### Preprocessor() ####
# log0_method              log1p                    str
# class_weights            (6,)                     Tensor (cuda:4)
# edge_index               (2, 32798)               Tensor (cuda:4)
# edge_attr                (32798, 16)              Tensor (cuda:4)
# gene_counts              (4383, 562)              DataFrame
# metadata                 (562, 3)                 DataFrame
# relation                 (32798, 18)              DataFrame
# node_id_map              4383                     dict
# mask_list                305                      list
# mask                     (4383, 305)              Tensor (cuda:4)
# x                        (562, 4383, 1)           Tensor (cuda:4)
# y                        (562,)                   Tensor (cuda:4)
# y_labels                 6                        list
# num_samples              562                      int
# num_nodes                4383                     int


In [2]:
#### convenience variables ####
_embedding_size = 16

# from mask (init)
_mask = brca.mask
_num_nodes, _num_sets = _mask.shape

# from batch (forward)
_batch_size = int(_batch.x.shape[0]/_num_nodes)
_num_node_features = _batch.x.shape[1] # or brca.num_node_features
_x = _batch.x.view(_batch.batch_size, int(_batch.x.shape[0]/_batch.batch_size), -1)

---

In [3]:
from modules.utils import reshape_x
from modules.model import cloneable, SequentialModel

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.nn import MessagePassing, GCNConv

from torch import Tensor
from torch_geometric.data import Batch
from typing import Literal, Optional, Union

In [4]:
def attn_dims(embed_dim:Optional[int]=None, head_dim:Optional[int]=None, num_heads:int=1):
    # none specified; assert error
    assert (embed_dim is not None) or (head_dim is not None), 'one of [embed_dim, head_dim] must be specified'

    # both specified; lin_out reshapes head to embed
    if (embed_dim is not None) and (head_dim is not None):
        assert embed_dim // num_heads == head_dim, 'transformer dims incompatible, (embed_dim // num_heads == head_dim) must be true'
        return embed_dim, head_dim

    # embed_dim specified; head = embed / num_heads
    elif embed_dim is not None:
        assert embed_dim % num_heads == 0, 'embed_dim must be divisible by num_heads'
        head_dim = embed_dim // num_heads

    # head_dim specified; embed = head * num_heads
    elif head_dim is not None:
        embed_dim = head_dim * num_heads

    return embed_dim, head_dim, num_heads

---

In [5]:
@cloneable
class TanhGATConv(MessagePassing):
    def __init__(self, in_channels:int, out_channels:int, edge_channels:Optional[int]=None, temperature:float=1.0, eps=1e-8, *args, **kwargs):
        super().__init__(aggr='add')
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.temperature = temperature
        self.eps = eps

        self.lin_q = nn.Linear(in_channels, out_channels)
        self.lin_kv = nn.Linear(in_channels, 2*out_channels)
        self.norm = nn.LayerNorm(out_channels)

        if edge_channels is not None:
            self.lin_e = nn.Linear(edge_channels, 6*out_channels)
        else:
            self.lin_e = None

    def forward(self, x:Tensor, edge_index:Tensor, edge_attr:Optional[Tensor]=None, *args, **kwargs):
        # get qkv
        q = self.lin_q(x)
        k, v = self.lin_kv(x).chunk(2, dim=-1)

        # transformer-like attention module
        out = self.propagate(edge_index, q=q, k=k, v=v, edge_attr=edge_attr)

        # add & norm
        out = self.norm(out + q)

        return out

    def message(self, q_i, k_j, v_j, edge_attr, index, size_i,):
        # get e_qkv (if applicable)
        if edge_attr is not None:
            # compute modulation values from edges
            q_e, k_e, v_e, q_gate, k_gate, v_gate = self.lin_e(edge_attr).chunk(6, dim=-1)

            # edge gated node modulation
            q_i = q_i + q_e * torch.sigmoid(q_gate)
            k_j = k_j + k_e * torch.sigmoid(k_gate)
            v_j = v_j + v_e * torch.sigmoid(v_gate)

        # vaswani dot product attention
        att = (q_i * k_j).sum(dim=-1) / self.out_channels

        # tanh L1 norm attention
        att = torch.tanh(att/self.temperature)
        att_sum = torch.zeros(size_i, device=att.device).scatter_add(0, index, att.abs())
        att = att / (att_sum[index] + self.eps)

        # save attention for analyses
        self._prev_att = att.detach()

        # apply attention to message
        return att.unsqueeze(-1) * v_j


---

In [6]:
@cloneable
class AttnSetPooling(nn.Module):
    def __init__(self, mask:Tensor, pool_method:Literal['add','mean']='mean', embed_dim:int=None, head_dim:int=None, num_heads:int=1, dropout:float=0.0, twin:bool=True, *args, **kwargs):
        '''
        mask in (nodes, sets)
        '''
        super().__init__(*args, **kwargs)
        self.embed_dim, _, _ = attn_dims(embed_dim, head_dim, num_heads)
        self.twin = twin

        # vars
        self.mask = mask.T # convert to (sets, nodes) for attn_mask in (q, kv)
        self.num_sets, self.num_nodes = self.mask.shape
        self.pool_method = pool_method
        self.set_counts = self.mask.sum(dim=-1, keepdim=True).clamp(min=1)  # sum across nodes, in (sets, 1); clamp to prevent div0
        
        # layers
        self.attn_layer = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, self.embed_dim),
            nn.ReLU(),
            nn.Linear(self.embed_dim, self.embed_dim)
        )
        self.norm2 = nn.LayerNorm(self.embed_dim)

    def attn(self, q, kv, need_weights, attn_mask=None):
        # get attention
        attn_out, weights = self.attn_layer(
            query=q,
            key=kv,
            value=kv,
            attn_mask = attn_mask,
            need_weights = need_weights
        )

        # get output
        h = self.norm1(attn_out + q) # add+norm (q is resid, before attn)
        ffn_out = self.ffn(h) # ffn
        h = self.norm2(ffn_out + h) # add+norm (h is resid, before ffn)

        return h, weights

    def forward(self, x:Tensor, need_weights:bool=False, as_dict:bool=True, *args, **kwargs):
        '''
        x in (b*n,f) or (b,n,f)
        attn_mask in (n, s) or (b * heads, n, s)
        '''
        # convert x to (b,n,F) if applicable
        x_node = reshape_x(x, 'b,n,f', num_nodes=self.num_nodes, num_node_features=self.embed_dim)

        # pool
        x_set = self.mask @ x_node # (b,s,n) @ (b,n,E) -> (b,s,E)
        if self.pool_method == 'mean':
            x_set = x_set / self.set_counts

        # get masked node-set attn
        self.attn_mask = (self.mask == 0)
        h_set, h_set_weights = self.attn(
            q = x_set,
            kv = x_node,
            attn_mask = self.attn_mask,
            need_weights = need_weights
        )

        # get node self-attn
        if self.twin:
            h_node, h_node_weights = self.attn(
                q = x_node,
                kv = x_node,
                need_weights = need_weights
            )
        else:
            h_node = None
            h_node_weights = None

        # format return
        if as_dict:
            return {'h_set':h_set, 'h_set_weights':h_set_weights, 'h_node':h_node, 'h_node_weights':h_node_weights}
        else:
            return h_set, h_set_weights, h_node, h_node_weights

In [7]:
# gat = TanhGATConv(
#     in_channels=brca.num_node_features,
#     out_channels=_embedding_size,
#     edge_channels=brca.num_edge_features
# )

# asp = AttnSetPooling(
#     mask=brca.mask,
#     head_dim=_embedding_size
# )

# _node_emb = gat(_batch.x, _batch.edge_index, _batch.edge_attr)

# _set_emb = asp(_node_emb, twin=True)
# _set_emb['h_set'].shape

---

In [8]:
@cloneable
class AttnGlobalPooling(nn.Module):
    def __init__(self, embed_dim:int=None, head_dim:int=None, num_heads:int=1, dropout:float=0.0, twin:bool=True, *args, **kwargs):
        super().__init__(*args, **kwargs)
        embed_dim, _, _ = attn_dims(embed_dim, head_dim, num_heads)
        self.twin = twin

        # cls-style global pooling tokens
        self.set_token = nn.Parameter(torch.randn(1,1,embed_dim))
        self.node_token = nn.Parameter(torch.randn(1,1,embed_dim)) if self.twin else None

        # layers
        self.attn_layer = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def attn(self, q, kv, need_weights):
        # get attention
        attn_out, weights = self.attn_layer(
            query=q, # cls
            key=kv, # x
            value=kv, # x
            need_weights = need_weights
        )

        # get output
        z = self.norm1(attn_out + q) # add+norm (q is resid, before attn)
        ffn_out = self.ffn(z) # ffn
        z = self.norm2(ffn_out + z) # add+norm (z is resid, before ffn)
        z = z.squeeze(1) # squeeze (b,1,E) -> (b,E)

        return z, weights

    def forward(self, h_set:Tensor, h_node:Tensor=None, need_weights:bool=False, as_dict:bool=True, *args, **kwargs):
        '''
        h_set in (b,s,E); opt h_node in (b,n,E); returns z in (b,E) via cls-token attention
        '''
        batch_size = h_set.shape[0]

        # expand token; get cross attn, in (b,1,E)
        q_set = self.set_token.expand(batch_size, -1, -1)
        z_set, z_set_weights = self.attn(
            q = q_set,
            kv = h_set,
            need_weights = need_weights
        )

        # get node self-attn
        if (self.twin) and (h_node is None):
            raise ValueError('h_node must be provided when self.twin=True')
        elif (self.twin) and (h_node is not None):
            q_node = self.node_token.expand(batch_size, -1, -1)
            z_node, z_node_weights = self.attn(
                q = q_node,
                kv = h_node,
                need_weights = need_weights
            )
        else:
            z_node = None
            z_node_weights = None

        # format return
        if as_dict:
            return {'z_set':z_set, 'z_set_weights':z_set_weights, 'z_node':z_node, 'z_node_weights':z_node_weights}
        else:
            return z_set, z_set_weights, z_node, z_node_weights

        

In [9]:
# tgat = TanhGATConv(
#     in_channels=brca.num_node_features,
#     out_channels=2*_embedding_size, # need to handle this in model
#     edge_channels=brca.num_edge_features
# )

# asp = AttnSetPooling(
#     mask=brca.mask,
#     head_dim=_embedding_size,
#     num_heads=2
# )

# agp = AttnGlobalPooling(
#     head_dim=_embedding_size,
#     num_heads=2
# )

# _node_emb = tgat(_batch.x, _batch.edge_index, _batch.edge_attr) # out (Tensor) = (b*n,F)
# _set_emb = asp(_node_emb) # out (dict) = h_ in (b,n,F) or (b,s,F); _weights in (n,n) or (s,s) ??
# _samp_emb = agp(_set_emb['h_node']) # out (dict) = z in (b, F)
# _samp_emb['z'].shape
# # # print(_samp_emb.shape)
# # _samp_emb

---

In [10]:
@cloneable
class EncoderOut(nn.Module):
    def __init__(self, embed_dim:int=None, head_dim:int=None, num_heads:int=1, hidden_dims:list[int]=[64], lin_kwargs:dict={}, twin:bool=False, vae:bool=False, *args, **kwargs):
        super().__init__(*args, **kwargs)
        embed_dim, _, _ = attn_dims(embed_dim, head_dim, num_heads)
        self.twin = twin
        self.vae = vae

        # reparameterize if VAE
        if self.vae:
            self.reparam = SequentialModel(
                in_channels=embed_dim,
                out_channels=2,
                layer_class=nn.Linear,
                hidden_dims=hidden_dims,
                **lin_kwargs
            )
        else:
            self.reparam = None

    def forward(self, z_set:Tensor, z_node:Optional[Tensor]=None, as_dict:bool=True, *args, **kwargs):
        # mean aggregate z if twin
        if (self.twin) and (z_node is None):
            raise ValueError('z_node must be provided when self.twin=True')
        elif (self.twin) and (z_node is not None):
            z = (z_set + z_node)/2
        else:
            z = z_set

        # reparam if vae
        if self.vae:
            z_mu, z_logvar = self.reparam(z).chunk(2, dim=-1)
            std = torch.exp(0.5 * z_logvar)
            z = z_mu + std * torch.randn_like(std)
        else:
            z_mu, z_logvar = None, None

        # format return
        if as_dict:
            return {'z':z, 'z_mu':z_mu, 'z_logvar':z_logvar}
        else:
            return z, z_mu, z_logvar

---

In [11]:
@cloneable
class Decoder(nn.Module):
    def __init__(self, num_nodes:int, embed_dim:int=None, head_dim:int=None, num_heads:int=1, hidden_dims:list[int]=[], lin_kwargs:dict={}, *args, **kwargs):
        super().__init__(*args, **kwargs)
        embed_dim, _, _ = attn_dims(embed_dim, head_dim, num_heads)
        self.num_nodes = num_nodes

        self.expand = SequentialModel(
            in_channels=embed_dim,
            out_channels=embed_dim * self.num_nodes,
            layer_class=nn.Linear,
            hidden_dims=hidden_dims,
            **lin_kwargs
        )

        self.estimate = SequentialModel(
            in_channels=embed_dim,
            out_channels=4,
            layer_class=nn.Linear,
            hidden_dims=hidden_dims,
            **lin_kwargs
        )

    def forward(self, z:Tensor, as_dict:bool=True, *args, **kwargs):
        '''
        reconstructs x and NB params. z in (b,E).
        '''
        batch_size = z.shape[0]

        # transform sample --> (b,n,E)
        z = self.expand(z).view(batch_size, self.num_nodes, -1)

        # estimate NB, FiLM params --> (b,n,4) --> 4 * (b,n)
        x_mu, x_theta, gate, mod = self.estimate(z).chunk(4, dim=-1)

        # NB, predict, resnet/film
        x_mu = torch.exp(x_mu)
        x_theta = torch.exp(x_theta)
        x_recon = x_mu + torch.sigmoid(gate) * mod

        # format return
        if as_dict:
            return {'x_recon':x_recon, 'x_mu':x_mu, 'x_theta':x_theta, 'gate':gate, 'mod':mod}
        else:
            return x_recon, x_mu, x_theta, gate, mod

---

In [12]:
@cloneable
class PathwayTransformer(nn.Module):
    def __init__(self, node_encoder:Union[nn.Module, MessagePassing], set_pool:nn.Module, global_pool:nn.Module, aggr:nn.Module, decoder:nn.Module, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.node_encoder = node_encoder.clone()
        self.set_pool = set_pool.clone()
        self.global_pool = global_pool.clone()
        self.aggr = aggr.clone()
        self.decoder = decoder.clone()

    def forward(self, x:Union[Tensor, Batch], as_dict:bool=True):
        # extract if batch
        if hasattr(x,'x'):
            batch = x
            x = batch.x
            # y = batch.y
            edge_index = batch.edge_index
            edge_attr = batch.edge_attr
            # laplacian_pe = batch.laplacian_pe
            batch_size = batch.batch_size
            num_node_features = batch.num_node_features

        # forward pass
        x=torch.log1p(x)
        h = self.node_encoder(x, edge_index=edge_index, edge_attr=edge_attr)
        h_set, _, h_node, _ = self.set_pool(h, as_dict=False)
        z_set, _, z_node, _ = self.global_pool(h_set, h_node, as_dict=False)
        z, _, _ = self.aggr(z_set, z_node, as_dict=False)
        x_recon, mu, theta, _, _ = self.decoder(z, as_dict=False)

        if as_dict:
            return {'x_recon':x_recon, 'mu':mu, 'theta':theta, 'z_set':z_set, 'z_node':z_node}
        else:
            return x_recon, mu, theta, z_set, z_node            


In [13]:
# tgat = TanhGATConv(
#     in_channels=brca.num_node_features,
#     out_channels=2*_embedding_size, # need to handle this in model
#     edge_channels=brca.num_edge_features
# )

# asp = AttnSetPooling(
#     mask=brca.mask,
#     head_dim=_embedding_size,
#     num_heads=2
# )

# agp = AttnGlobalPooling(
#     head_dim=_embedding_size,
#     num_heads=2
# )

# _node_emb = tgat(_batch.x, _batch.edge_index, _batch.edge_attr) # out (Tensor) = (b*n,F)
# _set_emb = asp(_node_emb) # out (dict) = h_ in (b,n,F) or (b,s,F); _weights in (n,n) or (s,s) ??
# _samp_emb = agp(_set_emb['h_node']) # out (dict) = z in (b, F)
# _samp_emb['z'].shape
# # # print(_samp_emb.shape)
# # _samp_emb

In [14]:
embed_dim, head_dim, num_heads = attn_dims(head_dim=16, num_heads=2)

gnn = SequentialModel(
    in_channels=brca.num_node_features,
    out_channels=embed_dim,
    layer_class=GCNConv,
)

set_pool = AttnSetPooling(
    mask=brca.mask,
    embed_dim=embed_dim,
)

global_pool = AttnGlobalPooling(
    embed_dim=embed_dim
)

aggr = EncoderOut(
    embed_dim=embed_dim
)

decoder = Decoder(
    num_nodes=brca.num_nodes,
    embed_dim=embed_dim,
)

model = PathwayTransformer(
    node_encoder=gnn,
    set_pool=set_pool,
    global_pool=global_pool,
    aggr=aggr,
    decoder=decoder
)

_out = model(_batch)


---

In [19]:
loader = t.Loader(
    dataset=_dataset,
    generator=generator,
    batch_size=16
)

trainer = t.NBTrainer(
    model=model,
    loader=loader,
    num_epochs=100,
    loss_fn=t.NBLoss(),
    optimizer_kwargs={'lr':5e-3},
    verbose=True,
    report_metrics=['loss','mae','rmse']
)

display(trainer.test_metrics)

for i in ['mu','theta']:
    for k,v in {'mean':np.mean, 'std':np.std, 'min':np.min, 'max':np.max}.items():
        print(f'({i}) {k}: {v(trainer.test_values[i])}')

 62%|██████▏   | 62/100 [00:31<00:19,  1.95it/s, Epoch 61      Train: loss=-10578132897185358938112.0000    mae=nan    rmse=nan        Val: loss=-10578109598563286646784.0000    mae=nan    rmse=nan]


KeyboardInterrupt: 